# Differential Privacy — Dev Log

## Objetivo

Mecanismo de Laplace real (`numpy.random.laplace`) para consultas agregadas
com privacidade diferencial — o primitivo clássico da literatura (Dwork et
al.), não uma simulação.

In [1]:
import sys
from pathlib import Path

REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from core.differential_privacy.mechanism import PrivacyBudget, private_count

records = list(range(1000))
real_count = sum(1 for r in records if r % 7 == 0)
result_low_eps = private_count(records, lambda r: r % 7 == 0, epsilon=0.1, seed=1)
result_high_eps = private_count(records, lambda r: r % 7 == 0, epsilon=10.0, seed=1)
print(f"Contagem real (múltiplos de 7 em 1000 registros): {real_count}")
print(f"Contagem privada (epsilon=0.1, mais privado): {result_low_eps.noisy_value:.2f}")
print(f"Contagem privada (epsilon=10.0, menos privado): {result_high_eps.noisy_value:.2f}")

budget = PrivacyBudget(total_epsilon=1.0)
budget.spend(0.3)
budget.spend(0.4)
print(f"Orçamento de privacidade: gasto={budget.spent}, restante={budget.remaining}")
try:
    budget.spend(0.5)
except ValueError as e:
    print(f"Tentativa de estourar o orçamento -> bloqueada: {e}")

Contagem real (múltiplos de 7 em 1000 registros): 143
Contagem privada (epsilon=0.1, mais privado): 143.24
Contagem privada (epsilon=10.0, menos privado): 143.00
Orçamento de privacidade: gasto=0.7, restante=0.3
Tentativa de estourar o orçamento -> bloqueada: Orçamento de privacidade excedido: tentando gastar 0.5, restam apenas 0.3.


Com `epsilon` alto (menos privacidade) o valor ruidoso já fica bem próximo
do real (143.00 vs 143 real); com `epsilon` baixo (mais privacidade) o ruído
é maior. `PrivacyBudget` impede estourar o orçamento acumulado de consultas.

## Testes e Handoff

```
"C:/Users/Yuri_/.venvs/athenagov-ai/Scripts/python.exe" -m pytest core/differential_privacy/tests -v
```

13/13 testes passando.